# 02 · Reconstrucción del flujo y process mining
Cada visita = caso · cada `visit_detail` = etapa. Calculamos duración de etapas y **esperas entre etapas** (el cuello de botella invisible).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import config, load, quality, flow, features, models

# Fuente de datos:
#   "db"     -> base de datos del reto (FortiClient VPN conectada + archivo .env)
#   "raw"    -> archivos exportados en data/raw
#   "sample" -> datos sintéticos de prueba (scripts/generar_datos_sinteticos.py)
SOURCE = "db"
DATA_DIR = ROOT / "data" / ("sample" if SOURCE == "sample" else "raw")

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
if SOURCE == "db":
    from src import db
    db.test_connection()
    t = db.load_all()   # copia local en data/interim; db.load_all(refresh=True) para volver a descargar
else:
    t = load.load_all(DATA_DIR)

In [ ]:
# Mapa de conceptos (concept_id -> nombre). Completar data/processed/concept_map.csv con los datos reales
cm_path = config.DATA_PROCESSED / "concept_map.csv"
if not cm_path.exists() and (DATA_DIR / "concept_map.csv").exists():
    cm_path = DATA_DIR / "concept_map.csv"
concept_names = (pd.read_csv(cm_path).set_index("concept_id")["concept_name"].to_dict()
                 if cm_path.exists() else {})
print(f"{len(concept_names)} conceptos mapeados")

In [ ]:
ev = flow.build_event_log(t['visit_detail'], concept_names)
ev[['visit_occurrence_id','orden','etapa','duracion_h','espera_previa_h','etapa_siguiente']].head(10)

## Duración por etapa

In [ ]:
st = flow.stage_stats(ev)
display(st)
st['p50_h'].plot.barh(figsize=(8,4), title='Mediana de duración por etapa (h)');

## Transiciones: ¿dónde se espera más?

In [ ]:
tr = flow.transitions(ev)
display(tr.head(15))
top = tr[tr['etapa_anterior']!='INICIO'].sort_values('espera_p50_h', ascending=False).head(10)
top.assign(t=top['etapa_anterior']+' → '+top['etapa']).set_index('t')['espera_p50_h'].plot.barh(figsize=(8,4), title='Top esperas entre etapas (mediana, h)');

## Variantes de trayectoria

In [ ]:
flow.variants(ev, top=15)

## Mapa de procesos con pm4py
(`pip install pm4py` y Graphviz instalado para visualizar)

In [ ]:
try:
    import pm4py
    log = pm4py.format_dataframe(flow.to_pm4py(ev), case_id='case:concept:name', activity_key='concept:name', timestamp_key='time:timestamp')
    dfg, sa, ea = pm4py.discover_dfg(log)
    pm4py.view_dfg(dfg, sa, ea)
    perf, sa, ea = pm4py.discover_performance_dfg(log)
    pm4py.view_performance_dfg(perf, sa, ea)
except Exception as e:
    print('pm4py/graphviz no disponible:', e)

## Guardar


In [ ]:
ev.to_parquet(config.DATA_INTERIM / 'event_log.parquet', index=False)
tr.to_csv(config.DATA_PROCESSED / 'transiciones.csv', index=False)
st.reset_index().to_csv(config.DATA_PROCESSED / 'duracion_etapas.csv', index=False)

## Hallazgos
- _(top 3 cuellos de botella con número)_